# LeakAudit - 01 - Load and cache the AI-Hub benchmark (D1)

**Parse the ten CSV files into arrays and a metadata table, then cache them.**

- Spectrum: 513 bins, 0-5120 Hz at 10 Hz spacing.
- Identifiers: 49 sites, 2 710 sensors; dates span 2019-2021.
- Device fields `lrate`/`llevel` are kept in metadata but never used as inputs.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # make 'src' importable from notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation
cfg.apply_mpl_style()

In [2]:
# Load (and cache) the tabular benchmark.
X1, X1_max, M1, HZ = io_utils.load_d1(force=False)

print(f"rows        : {len(M1):,}")
print(f"spectrum    : {X1.shape[1]} bins, {HZ.min()}-{HZ.max()} Hz, step {HZ[1]-HZ[0]} Hz")
print(f"repeat cols : {X1_max.shape[1]}")
print(f"sites       : {M1.site.nunique()}   sensors: {M1.sid.nunique()}")
print(f"dates       : {M1.dt.min().date()} - {M1.dt.max().date()}")
print()
print(pd.crosstab(M1.cls, M1.split, margins=True).to_string())

c:\Users\ecredible\Desktop\LeakAudit\LeakAudit\src\io_utils.py:132: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(cls=_d1_class_of(path), split=_d1_split_of(path))


rows        : 70,384
spectrum    : 513 bins, 0-5120 Hz, step 10 Hz
repeat cols : 20
sites       : 49   sensors: 2710
dates       : 2019-02-27 - 2021-02-10

split   train   val    All
cls                       
in      13273  1659  14932
noise    5029   629   5658
normal  19704  2462  22166
other    7019   878   7897
out     17539  2192  19731
All     62564  7820  70384


The cache now lives in `data/`; later notebooks reload it in a second.

In [3]:
print("cached files:")
for p in sorted(cfg.CACHE.glob("d1_*")):
    print("  ", p.name, f"{p.stat().st_size/1e6:.1f} MB")

cached files:
   d1_hz.npy 0.0 MB
   d1_max.npy 5.6 MB
   d1_meta.csv 5.4 MB
   d1_spec.npy 144.4 MB
